# Reimplementazione

## Il problema
Definire una PINN che risolve l'equazione di Fokker Planck $ \nabla \cdot (-fp + \nabla \cdot (\Sigma p)) = p_t$

Dove $f:\mathbb{R}^n\times\mathbb{R}_+ \to \mathbb{R}^m$, $\Sigma:\mathbb{R}^n\times\mathbb{R}_+ \to \mathbb{R}^{m\times n}$, $p:\mathbb{R}^n\times\mathbb{R}_+ \to \mathbb{R}$ and $p_t:\mathbb{R}^n\times\mathbb{R}_+ \to \mathbb{R}$

In particolare la divergenza di una matrice agisce parallelamente.

Inoltre, si richide $\forall t \left[\int p(x,t)dx=1\right]$. Questo è anche sufficiente per avere $p\geq0$

## Metodologia

Usare una Rete Neurale che considera come loss unicamente la PDE e, eventualmente, i parametri della rete. La perdita dedotta dalla PDE del modello proposto sarà:
$$
    \left\|\mathcal{L}_\text{PDE}(\tilde{p})-p_t\right\|_{L^2}^2 + \lambda \left(\int \tilde{p}(x,t) dx-1\right)^2 \quad \lambda > 0
$$
Osserviamo per prima cosa che questo è un problema convesso nell'insieme delle funzioni $\tilde{p}$, senza considerare la parametrizzazione, in quanto $\mathcal{L}_\text{PDE}$ è un operatore lineare, inoltre il suo minimo globale è chiaramente una delle soluzioni desiderate (potrebbe essercene più di una).

Mentre la porzione di loss data dai parametri della rete neurale serve per stabilizzare il training, ma richiede l'architettura della rete per comprenderne il significato.

## La rete neurale

Solitamente le architetture più promettenti sono le classiche MLP. Ossia un concatenamento di $\sigma(Ax+b)$ dove $\sigma$ è la funzione di attivazione (non lineare e solitamente con saturazione in almeno una coda) che agisce singolarmente su ogni feature (o neurone, o componente), $x\in\mathbb{R}^n$ $A\in\mathbb{R}^{m\times n}$, $b\in\mathbb{R}^m$. Questa struttura è detta Layer ed esiste in letteratura solo un'altra famiglia di layer: Gated Linear Unit definita come: $\sigma(A_1x+b_1)\otimes(A_2x+b_2)$ dove $\otimes$ è il prodotto per feature.

Le funzioni di attivazione possono essere di vari tipi:
- doppia coda saturata: $\tanh$, $\text{sigmoid}$
- singola coda saturata: $\text{SiLU}$, $\text{GELU}$, $\text{Mish}$, $\text{SoftPlus}$
- asimmetrico: $\text{LeakySoftPlus}$

Nota: Il layer $\text{GLU}$ è molto più complesso ma anche flessibile.

### Lipschitzianeità
Un layer lipschitziano è molto comodo poiché preserva comportamenti polinomiali. Qualora un layer non fosse lipschitziano, ma semplicemente polinomiale, nulla vieta di fare quanto segue:
$$
    \text{Closure} \circ \text{LayerLip} \circ \cdots \text{LayerLip} \circ \text{LayerPol}
$$
dove $\text{Closure}$ regola l'output (ad esempio $x\mapsto \left(1+x^2\right)^{1/4}$ se il grado interno è $2$)

Ci sono anche altri sistemi come $\text{LofSoftMax}$ che amplifica invece di ridurre, ma rimane comunque lipschitziano.

In [7]:
import torch.nn as nn

# definisco una classica MLP di ampiezza 64
class MLP(nn.Module):
    class subMLP(nn.Module):
        def __init__(self, in_features, out_features, deep):
            super(MLP.subMLP, self).__init__()
            net_list: list[nn.Module] = []
            k = (out_features / in_features) ** (1 / deep)
            in_curr = in_features
            for i in range(deep):
                out_curr = int(in_features * k ** (i + 1)) if i < deep - 1 else out_features
                net_list.append(
                    nn.Sequential(
                        nn.Linear(in_curr, out_curr),
                        nn.Tanh()
                    )
                )
                in_curr = out_curr
            self.net_list = nn.ModuleList(net_list)
        def forward(self, x):
            for layer in self.net_list:
                x = layer(x)
            return x

    def __init__(self):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            # opening
            self.subMLP(1, 64, 4),
            # stasis
            self.subMLP(64, 64, 3),
            # closing
            self.subMLP(64, 1, 4)
        )

    def forward(self, x):
        return self.net(x)

Per implementare la loss vorrei poter calcolare parallelamente la jacobiana e l'hessiana

In [ ]:
from torch.func import vmap, jacrev, hessian

def compute_jacobian(model, x):
    # x is of shape (batch_size, input_dim)
    # model outputs shape (batch_size, output_dim)
    J = vmap(jacrev(model))(x)
    return J

def compute_hessian(model, x):
    # x is of shape (batch_size, input_dim)
    # model outputs shape (batch_size, output_dim)
    H = vmap(hessian(model))(x)
    return H

In [ ]:
# Example usage:
if __name__ == "__main__":
    import torch
    model = MLP()
    x = torch.randn(10, 1)  # batch of 10 samples, input dimension 1
    jacobian = compute_jacobian(model, x)
    hessian = compute_hessian(model, x)
    print("Jacobian shape:", jacobian.shape)  # should be (10, output_dim, input_dim)
    print("Hessian shape:", hessian.shape)    # should be (10, output_dim, input_dim, input_dim)

Jacobian shape: torch.Size([10, 1, 1])
Hessian shape: torch.Size([10, 1, 1, 1])


Pensiamo di risolvere il semplice processo OU con equazione di Fokker Planck: $ \nabla\cdot\left(-fp + \nabla\cdot\left(\Sigma p\right)\right) = p_t $

dove: $ f(x,t) = -\nabla U(x) = -\nabla \frac{1}{2}\left\|x\right\|^2 = -x \;,\quad \Sigma(x,t) = \sigma^2 \text{Id} $

Per semplicità di calcolo si può riscrivere l'operatore lineare come segue:
$$
    \nabla\cdot\left(-fp + \nabla\cdot\left(\Sigma p\right)\right) = \left(-\nabla \cdot f\right)p + \left(-f\right)\cdot\nabla p + \nabla \cdot \left(\left(\nabla \cdot \Sigma \right)p+\Sigma\cdot\nabla p\right)
$$
$$
    = \left(-\nabla \cdot f\right)p + \left(-f\right)\cdot\nabla p + \nabla \cdot \left(\left(\nabla \cdot \Sigma \right)p\right)+\nabla\cdot\left(\Sigma\cdot\nabla p\right) 
$$
$$
    = \left(-\nabla \cdot f\right)p + \left(-f\right)\cdot\nabla p + \left(\nabla \cdot \nabla \cdot \Sigma\right) p+ \left(\nabla \cdot \Sigma\right)\cdot \nabla p + \nabla\cdot\left(\Sigma\cdot\nabla p\right) 
$$

In [ ]:
# implementazione loss function

